# Step 6 – Sylheti Voice Cloning (MMS-TTS fine-tune, speaker V1)

**Runtime → Change runtime type → T4 GPU (বা L4)**, তারপর **Run all**।  
সম্ভব হলে **0621 অ্যাকাউন্ট দিয়ে চালান** (তাহলে সব ফাইল 0621-এর নামে থাকবে)।

কী হবে (সব ধাপ আবার চালালে আগের জায়গা থেকে চলবে):
1. দুই ফোল্ডার কপি → `Sylheti_Thesis/voice_clone/raw_V1/` (মূল ফোল্ডারে কিছু বদলাবে না, কিছু মুছবে না)
2. ডেটা তৈরি: নতুন web-recorder রেকর্ডিং + Round 1-এর A4 (৫৫০১–৬০০০) → 16 kHz WAV, সঠিক সিলেটি লেখা
3. Quality check: fine-tuned Whisper-turbo দিয়ে লিখে মেলানো; খারাপ/ভুল-বাক্যের রেকর্ডিং বাদ
4. `facebook/mms-tts-ben` fine-tune (`finetune-hf-vits`) — checkpoint Drive-এ সেভ হয়, disconnect হলে Run all দিলে সেখান থেকে চলবে
5. মূল্যায়ন: আগের MMS বনাম নতুন মডেল (শুনে দেখা + CER)

- প্রথমবার install-এর পর session নিজে restart হবে ("session crashed") — তখন আবার **Run all**।
- Section 1-এ Google অ্যাকাউন্টের অনুমতি চাইবে (ফোল্ডার কপির জন্য) — Allow দিন।
- Speaker নাম কোথাও লেখা হয় না; কোড **V1**।

## 0. Library আর finetune-hf-vits

In [ ]:
import os, sys, subprocess
from importlib.metadata import version, PackageNotFoundError
NEED = {"transformers": "4.48.3", "datasets": "3.6.0"}
def _ver(p):
    try:
        return version(p)
    except PackageNotFoundError:
        return None
if any(_ver(p) != v for p, v in NEED.items()):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.48.3", "datasets[audio]==3.6.0",
                    "accelerate", "Cython", "tensorboard", "matplotlib", "jiwer", "sentencepiece"], check=True)
    print("installed -> restarting the session; press Run all again")
    os.kill(os.getpid(), 9)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jiwer", "Cython"], check=True)

REPO = "/content/finetune-hf-vits"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "https://github.com/ylacombe/finetune-hf-vits", REPO], check=True)
MA = REPO + "/monotonic_align"
if not any(f.endswith(".so") for _, _, fs in os.walk(MA) for f in fs):
    os.makedirs(MA + "/monotonic_align", exist_ok=True)
    subprocess.run([sys.executable, "setup.py", "build_ext", "--inplace"], cwd=MA, check=True, capture_output=True)
import torch
assert torch.cuda.is_available(), "GPU নেই: Runtime -> Change runtime type -> T4 GPU (training-এর জন্য GPU লাগবেই)"
print(torch.cuda.get_device_name(0), "| transformers", version("transformers"), "| datasets", version("datasets"))

## 1. Drive, ঠিকানা আর ফোল্ডার কপি

In [ ]:
import glob, shutil, json, re, unicodedata, time
import numpy as np, pandas as pd
from google.colab import drive, auth
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
auth.authenticate_user()                       # for server-side folder copy (Drive API)
from googleapiclient.discovery import build
svc = build("drive", "v3", cache_discovery=False)

BASE = "/content/drive/MyDrive/Sylheti_Thesis"
ROOT = BASE + "/Ultimate Thisis Data"
ASR_DIR = BASE + "/sylheti_asr"
VC = BASE + "/voice_clone"                     # created by Claude, owned by 0621
OUT_MODEL = VC + "/model_V1_mms-ben"
DISC = VC + "/mms-tts-ben-train"               # base model + converted discriminator (made once)
EVAL = VC + "/eval_V1"
os.makedirs(VC, exist_ok=True)

SRC = {   # Drive folder ids: source -> copy target (inside voice_clone/raw_V1)
    "web_recorder_sylheti": ("1Pj4-2CjzshIvWpANU8-NKmk2uNxh5Hga", "1_XLOD8NUgoajs0FEl2nvol_CJKcpq6oT"),
    "A4_5501-6000_m4a":     ("1o7ZzjaZlHskYn_me4YUdePfOM6jm67hZ", "13pWJt_EVaHUjaSwRG13HFCYAlzvTxO_C"),
}

def children(fid):
    out, tok = [], None
    while True:
        r = svc.files().list(q=f"'{fid}' in parents and trashed=false", pageSize=1000, pageToken=tok,
                             fields="nextPageToken, files(id,name,mimeType,size)",
                             supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
        out += r["files"]; tok = r.get("nextPageToken")
        if not tok:
            return out

for name, (src, dst) in SRC.items():
    have = {f["name"] for f in children(dst)}
    todo = [f for f in children(src) if f["mimeType"] != "application/vnd.google-apps.folder" and f["name"] not in have]
    for i in range(0, len(todo), 100):          # batch of 100 copy requests
        b = svc.new_batch_http_request()
        for f in todo[i:i + 100]:
            b.add(svc.files().copy(fileId=f["id"], body={"name": f["name"], "parents": [dst]}, supportsAllDrives=True))
        b.execute()
    print(f"{name}: {len(have)} আগে ছিল, {len(todo)} নতুন কপি -> মোট {len(children(dst))}")

## 2. ডেটা তৈরি (16 kHz WAV + সিলেটি লেখা)

In [ ]:
import tarfile, io, soundfile as sf
from concurrent.futures import ThreadPoolExecutor
from googleapiclient.http import MediaIoBaseDownload
import google.auth

W = "/content/v1"; os.makedirs(W + "/new_raw", exist_ok=True); os.makedirs(W + "/new_wav", exist_ok=True)
corpus = pd.read_csv(ROOT + "/corpus.csv", encoding="utf-8-sig").set_index("excel_row")

# --- (a) new web-recorder clips: download the copies, convert with ffmpeg (16 kHz mono, trim silence) ---
new_files = [f for f in children(SRC["web_recorder_sylheti"][1]) if re.fullmatch(r"syl_\d+\.\w+", f["name"])]
import threading
creds, _ = google.auth.default()
_tl = threading.local()
def _svc():
    if not hasattr(_tl, "s"):
        _tl.s = build("drive", "v3", credentials=creds, cache_discovery=False)
    return _tl.s
def fetch(f):
    raw = f"{W}/new_raw/{f['name']}"
    wav = f"{W}/new_wav/{os.path.splitext(f['name'])[0]}.wav"
    if not os.path.exists(wav):
        if not os.path.exists(raw):
            s = _svc()
            with io.FileIO(raw, "wb") as fh:
                d = MediaIoBaseDownload(fh, s.files().get_media(fileId=f["id"], supportsAllDrives=True))
                done = False
                while not done:
                    _, done = d.next_chunk()
        trim = ("silenceremove=start_periods=1:start_threshold=-45dB:start_silence=0.1,areverse,"
                "silenceremove=start_periods=1:start_threshold=-45dB:start_silence=0.1,areverse")
        subprocess.run(["ffmpeg", "-nostdin", "-loglevel", "error", "-y", "-i", raw, "-ac", "1", "-ar", "16000",
                        "-af", trim, wav], check=False)
    return wav
with ThreadPoolExecutor(8) as ex:
    wavs = list(ex.map(fetch, new_files))
new = pd.DataFrame({"path": wavs, "num": [int(re.findall(r"\d+", os.path.basename(w))[0]) for w in wavs]})
new = new[new.path.map(os.path.exists)]
new["source"] = "web_recorder"
print("web-recorder clips:", len(new), "| বাক্য নম্বর", new.num.min(), "-", new.num.max())

# --- (b) Round 1 A4, sentences 5501-6000: already converted (wav_16k) and corrected in Step 1c ---
mp = pd.read_csv(ROOT + "/mapping.csv", encoding="utf-8-sig")
a4_names = set(mp[(mp.speaker == "A4") & mp.original_file.astype(str).str.contains("5501-6000")].wav_name)
meta = pd.read_csv(ASR_DIR + "/metadata.csv")
meta["wav_name"] = meta.audio_path.str.split("/wav_16k/").str[-1]
a4 = meta[meta.wav_name.isin(a4_names)].copy()      # clips dropped in Step 1c are not in metadata
OUT16 = "/content/wav_16k"
if not all(os.path.exists(f"{OUT16}/{w}") for w in a4.wav_name):
    print("extracting A4.tar ...")
    with tarfile.open(ROOT + "/wav_16k_tar/A4.tar") as tf:
        tf.extractall(OUT16)
a4 = pd.DataFrame({"path": OUT16 + "/" + a4.wav_name, "num": a4.excel_row.astype(int), "source": "A4_round1"})
a4 = a4[a4.path.map(os.path.exists)]
print("A4 (5501-6000) clips:", len(a4))

## 3. Quality check (fine-tuned Whisper-turbo)
নতুন রেকর্ডিংয়ের ফাইল নম্বর আর corpus-এর বাক্য নম্বর মেলে কিনা (offset) দেখে, তারপর প্রতিটা রেকর্ডিংয়ের CER মাপে।

In [ ]:
import jiwer
from transformers import WhisperProcessor, WhisperForConditionalGeneration
ZW = dict.fromkeys(map(ord, "\u200b\u200c\ufeff\u00ad\u2060"), None)
def clean(s):
    s = unicodedata.normalize("NFC", str(s)).translate(ZW)
    s = re.sub("\u200d(?!\u09cd\u09af)", "", s)
    s = re.sub(r"[\"\'“”‘’(),.।?!:;\-/*…\[\]|]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

ASR_MODEL = next(p for p in [ASR_DIR + "/whisper-large-v3-turbo-4spk/final",
                             *sorted(glob.glob(BASE + "/0621_copy/*turbo*/final"))] if os.path.exists(p))
proc = WhisperProcessor.from_pretrained(ASR_MODEL, language="bengali", task="transcribe")
asr = WhisperForConditionalGeneration.from_pretrained(ASR_MODEL, torch_dtype=torch.float16).to("cuda").eval()
asr.generation_config.language, asr.generation_config.task = "bengali", "transcribe"
asr.generation_config.forced_decoder_ids = None

@torch.no_grad()
def transcribe(paths, bs=16):
    out = []
    for i in range(0, len(paths), bs):
        auds = [sf.read(p, dtype="float32")[0][: 30 * 16000] for p in paths[i:i + bs]]
        x = proc.feature_extractor(auds, sampling_rate=16000, return_tensors="pt").input_features
        ids = asr.generate(x.to("cuda", torch.float16), max_new_tokens=225)
        out += proc.tokenizer.batch_decode(ids, skip_special_tokens=True)
    return out

def cer(hyp, row):
    ref = clean(corpus.sylheti_asr.get(row, ""))
    return jiwer.cer(ref, clean(hyp) or "-") if ref else 9.9

QC = VC + "/quality_report_V1.csv"
if os.path.exists(QC):
    qc = pd.read_csv(QC, encoding="utf-8-sig")
else:
    allc = pd.concat([new, a4], ignore_index=True)
    allc["dur"] = allc.path.map(lambda p: sf.info(p).duration)
    allc["asr"] = transcribe(allc.path.tolist())
    # offset check for web-recorder numbering (file number vs corpus excel_row)
    n = allc[allc.source == "web_recorder"].head(200)
    offs = {k: float(np.median([cer(h, r + k) for h, r in zip(n.asr, n.num)])) for k in range(-3, 4)}
    best = min(offs, key=offs.get)
    print("offset -> median CER:", {k: round(v, 3) for k, v in offs.items()}, "| বেছে নেওয়া offset:", best)
    allc["excel_row"] = allc.num + np.where(allc.source == "web_recorder", best, 0)
    allc["cer"] = [cer(h, r) for h, r in zip(allc.asr, allc.excel_row)]
    allc["text"] = allc.excel_row.map(corpus.sylheti)
    allc["split"] = allc.excel_row.map(corpus.split)
    allc.to_csv(QC, index=False, encoding="utf-8-sig"); qc = allc
del asr; torch.cuda.empty_cache()

MAX_CER = 0.5          # lenient: Sylheti spelling varies; turbo's average CER on A4 was ~0.20
keep = qc[(qc.cer <= MAX_CER) & qc.dur.between(1.0, 20.0) & qc.text.notna()]
print(qc.groupby("source").agg(clips=("cer", "size"), median_cer=("cer", "median")).round(3).to_string())
print(f"রাখা হলো {len(keep)} / {len(qc)}  (CER ≤ {MAX_CER}, 1–20 s)  | মোট {keep.dur.sum()/3600:.2f} ঘণ্টা")
print("report:", QC)

## 4. Train / eval ভাগ আর dataset ফোল্ডার

In [ ]:
DS = "/content/ds_v1"
shutil.rmtree(DS, ignore_errors=True)
def tts_text(s):
    s = unicodedata.normalize("NFC", str(s)).translate(ZW)
    s = re.sub(r"[\"“”‘’()\[\]{}*]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s if s.endswith(("।", "?", "!")) else s + "।"

# eval: up to 50 web-recorder clips whose sentence is in the MT/ASR test split (never in training)
cand = keep[(keep.source == "web_recorder") & (keep.split == "test")]
if len(cand) < 50:
    cand = pd.concat([cand, keep[(keep.source == "web_recorder") & ~keep.index.isin(cand.index)]])
ev = cand.drop_duplicates("excel_row").sample(min(50, len(cand)), random_state=13)
tr = keep[~keep.excel_row.isin(set(ev.excel_row))]
for name, d in [("train", tr), ("test", ev)]:
    os.makedirs(f"{DS}/{name}", exist_ok=True)
    rows = []
    for i, r in enumerate(d.itertuples()):
        fn = f"{r.source}_{r.excel_row}_{i}.wav"
        shutil.copy(r.path, f"{DS}/{name}/{fn}")
        rows.append({"file_name": fn, "text": tts_text(r.text)})
    pd.DataFrame(rows).to_csv(f"{DS}/{name}/metadata.csv", index=False, encoding="utf-8")
ev[["source", "excel_row", "text", "path"]].to_csv(VC + "/eval_sentences_V1.csv", index=False, encoding="utf-8-sig")
print("train:", len(tr), "| eval:", len(ev))

## 5. Discriminator তৈরি (একবারই, ~৫ মিনিট)
Fine-tune করতে MMS-এর discriminator লাগে; `convert_original_discriminator_checkpoint.py` সেটা Hugging Face থেকে নিয়ে বানায়। একবার বানালে Drive-এ থাকে।

In [ ]:
if not os.path.exists(DISC + "/config.json"):
    subprocess.run([sys.executable, "convert_original_discriminator_checkpoint.py", "--language_code", "ben",
                    "--pytorch_dump_folder_path", "/content/mms-tts-ben-train"], cwd=REPO, check=True)
    shutil.copytree("/content/mms-tts-ben-train", DISC, dirs_exist_ok=True)
print("base for fine-tuning:", DISC, os.listdir(DISC))

In [ ]:
!pip uninstall -y -q torchvision
p = REPO + "/utils/plot.py"
s = open(p).read().replace("np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)",
                           "np.asarray(fig.canvas.buffer_rgba())[..., :3].copy()")
open(p, "w").write(s)
print("plot.py fixed:", "tostring_rgb" not in s)

## 6. Fine-tune (T4-এ আনুমানিক ১.৫–৩ ঘণ্টা)
প্রতি ৫০০ step-এ checkpoint Drive-এ সেভ হয় (সর্বোচ্চ ২টা)। Disconnect হলে আবার **Run all** দিন — শেষ checkpoint থেকে চলবে।

In [ ]:
from collections import deque
cfg = json.load(open(VC + "/train_config_V1.json"))
cfg["preprocessing_num_workers"] = None
json.dump(cfg, open(VC + "/train_config_V1.json", "w"), ensure_ascii=False, indent=1)

env = dict(os.environ, WANDB_MODE="disabled", PYTHONUNBUFFERED="1", ACCELERATE_MIXED_PRECISION="fp16")
p = subprocess.Popen([sys.executable, "run_vits_finetuning.py", VC + "/train_config_V1.json"],
                     cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
tail = deque(maxlen=80)
for line in p.stdout:
    tail.append(line.rstrip())
    if "Steps" in line or "Saved" in line:
        print(line.rstrip()[-200:])
if p.wait() != 0:
    print("\n".join(tail))
    raise SystemExit("training failed – উপরের লেখার screenshot দিন")
print("done:", OUT_MODEL)


## 7. মূল্যায়ন: আগের MMS বনাম নতুন (V1)
Eval বাক্যগুলো (training-এ ছিল না) দুই মডেলে বলিয়ে fine-tuned Whisper-turbo দিয়ে আবার লেখায় এনে CER মাপে। অডিও Drive-এ `voice_clone/eval_V1`-এ সেভ হয়।

In [ ]:
from transformers import VitsModel, AutoTokenizer
from IPython.display import Audio, display
asr = WhisperForConditionalGeneration.from_pretrained(ASR_MODEL, torch_dtype=torch.float16).to("cuda").eval()
asr.generation_config.language, asr.generation_config.task = "bengali", "transcribe"
asr.generation_config.forced_decoder_ids = None
os.makedirs(EVAL, exist_ok=True)
models = {"mms_base": "facebook/mms-tts-ben", "V1_finetuned": OUT_MODEL}
rows = []
for tag, path in models.items():
    tok = AutoTokenizer.from_pretrained(path); m = VitsModel.from_pretrained(path).to("cuda").eval()
    paths = []
    for i, r in enumerate(ev.itertuples()):
        with torch.no_grad():
            w = m(**tok(tts_text(r.text), return_tensors="pt").to("cuda")).waveform[0].float().cpu().numpy()
        f = f"{EVAL}/{tag}_{r.excel_row}.wav"; sf.write(f, w, m.config.sampling_rate); paths.append(f)
    hyps = transcribe(paths)
    rows.append({"model": tag, "CER (fine-tuned ASR)": round(100 * np.mean([cer(h, r) for h, r in zip(hyps, ev.excel_row)]), 1),
                 "clips": len(paths)})
    del m; torch.cuda.empty_cache()
res = pd.DataFrame(rows); res.to_csv(EVAL + "/results.csv", index=False)
print(res.to_string(index=False))
print("\nতুলনার জন্য: আসল রেকর্ডিংয়ের CER (একই বাক্য):", round(100 * ev.cer.mean(), 1))
for r in ev.head(3).itertuples():
    print("\n", r.text)
    for tag in models:
        print(tag); display(Audio(f"{EVAL}/{tag}_{r.excel_row}.wav"))
    print("real recording"); display(Audio(r.path))

---
**নোট**
- নতুন মডেল সিলেটি লেখা থেকে V1-এর কণ্ঠে সিলেটি বলে; বাংলা লেখা দিলে একই কণ্ঠে সিলেটি টানে বাংলা বলবে।
- CER কম হলে বোঝা যায় কথাটা বোঝা যাচ্ছে; কণ্ঠ কতটা মিলছে সেটা কানে শুনে (বা MOS দিয়ে) বিচার করতে হবে।
- Demo-তে (Step 5) বাংলা TTS-এর মডেল `facebook/mms-tts-ben` থেকে `voice_clone/model_V1_mms-ben`-এ বদলালেই নতুন কণ্ঠ আসবে।
- কোনো সেলে error এলে screenshot পাঠান।